In [ ]:
# Script plots ERA5 extreme heat season start and end date changes. It can be used to re-create ERA5 figures in the manuscript.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import os
import re
import pandas as pd
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks

In [ ]:
# Load in the ERA5 Heat Season Characteristics Files for the two time periods of interest.
# Script is designed to work with one temperature variable (TMAX or TMIN) at a time.

ds1 = xr.load_dataset("/...baseline period TMAX file...")
ds2 = xr.load_dataset("/...comparison period TMAX file...")


In [ ]:
# Grab the array of analytical year start months (see Methods in the paper).
trop_map_p1 = ds1['ref_trop_start_month']

In [ ]:
def calculate_true_season_shift(da_p1, da_p2, trop_start_month_map):
    """
    Converts Day of Year (DOY) to Day of Season (DOS) using the local start month.
    Automatically handles global data by applying NH/SH rules to extra-tropics.
    """
    # 1. Force alignment of P2 to P1 to prevent floating point coordinate errors
    da_p2_aligned = da_p2.reindex_like(da_p1, method='nearest', tolerance=0.01)
    
    # 2. Build a Global Start Month Map matching da_p1
    # Default Extra-Tropical rules: Northern Hemisphere = Month 1, Southern = Month 7 (see Manuscript Methods)
    global_start_month = xr.where(da_p1.lat >= 0, 1, 7)
    
    # 3. Align the Tropical map and layer it on top
    trop_aligned = trop_start_month_map.reindex_like(da_p1, method='nearest', tolerance=0.01)
    global_start_month = xr.where(trop_aligned.notnull(), trop_aligned, global_start_month)
    
    # 4. Calculate approximate start DOY boundary
    approx_start_doy = (global_start_month - 1) * 30
    
    # 5. Convert DOY to DOS (Day of Season)
    # If a day falls numerically before the season boundary, push it into "Year 2"
    dos_p1 = xr.where(da_p1 < approx_start_doy, da_p1 + 365.25, da_p1)
    dos_p2 = xr.where(da_p2_aligned < approx_start_doy, da_p2_aligned + 365.25, da_p2_aligned)
    
    # 6. Calculate true arithmetic means on this unrolled, linear timeline
    mean_dos_p1 = dos_p1.mean(dim='year', skipna=True)
    mean_dos_p2 = dos_p2.mean(dim='year', skipna=True)
    
    # 7. Calculate the true shift
    true_diff = mean_dos_p2 - mean_dos_p1
    
    return true_diff

In [ ]:
# ==========================================
# CALCULATE TRUE SHIFTS (Starts & Ends)
# ==========================================
# Starts
shift_s1_start = calculate_true_season_shift(ds1['s1_start'], ds2['s1_start'], ds1['ref_trop_start_month'])
shift_s2_start = calculate_true_season_shift(ds1['s2_start'], ds2['s2_start'], ds2['ref_trop_start_month'])

# Ends
shift_s1_end = calculate_true_season_shift(ds1['s1_end'], ds2['s1_end'], ds1['ref_trop_start_month'])
shift_s2_end = calculate_true_season_shift(ds1['s2_end'], ds2['s2_end'], ds2['ref_trop_start_month'])

In [ ]:
# ==========================================
# AVERAGE THE BIMODAL SHIFTS
# ==========================================
# Average the two shifts together
avg_start_shift = (shift_s1_start + shift_s2_start) / 2
avg_end_shift = (shift_s1_end + shift_s2_end) / 2

# Apply the average ONLY where Season 2 exists. Otherwise, keep Season 1 shift.
final_start_shift = xr.where(shift_s2_start.notnull(), avg_start_shift, shift_s1_start)
final_end_shift = xr.where(shift_s2_end.notnull(), avg_end_shift, shift_s1_end)

In [ ]:
def circular_permutation_test(da_p1, da_p2, n_permutations=1000, days_in_year=365.25):
    """
    Performs a permutation test on CIRCULAR data (Day of Year).
    Tests if the mean direction of P1 is significantly different from P2.
    
    Args:
        da_p1 (xr.DataArray): Time series for Period 1 (lat x lon x year)
        da_p2 (xr.DataArray): Time series for Period 2 (lat x lon x year)
        n_permutations (int): Number of shuffles (default 1000)
        days_in_year (float): Cycle length (default 365.25)
        
    Returns:
        p_value_map (xr.DataArray): Map of p-values (0.0 to 1.0)
        obs_diff_days (xr.DataArray): The observed change in days (P2 - P1)
    """
    print(f"Starting Circular Permutation Test ({n_permutations} permutations)...")

# --- 1. PREPARE DATA ---
    # Combine data to shuffle effectively
    # Add .transpose(..., 'year') to FORCE year to be the very last axis
    combined = xr.concat([da_p1, da_p2], dim='year').transpose(..., 'year')
    
    n_p1 = da_p1.sizes['year']
    n_total = combined.sizes['year']
    
    
    # Convert ENTIRE dataset to Radians (Vectors) once
    # 0 to 2pi
    angles = (combined - 1) * (2 * np.pi / days_in_year)
    sin_vals = np.sin(angles)
    cos_vals = np.cos(angles)
    
    # --- 2. CALCULATE OBSERVED STATISTIC ---
    def get_circular_mean(s, c, dim):
        # Mean Vector
        s_bar = s.mean(dim=dim)
        c_bar = c.mean(dim=dim)
        # Mean Angle
        return np.arctan2(s_bar, c_bar)

    # Split back into P1/P2 based on original indices
    # We select by integer index (isel)
    obs_s_p1 = sin_vals.isel(year=slice(0, n_p1))
    obs_c_p1 = cos_vals.isel(year=slice(0, n_p1))
    obs_s_p2 = sin_vals.isel(year=slice(n_p1, n_total))
    obs_c_p2 = cos_vals.isel(year=slice(n_p1, n_total))
    
    # Calculate Mean Angles for P1 and P2
    # Note: We skip NaNs automatically with xarray mean
    mean_angle_p1 = get_circular_mean(obs_s_p1, obs_c_p1, 'year')
    mean_angle_p2 = get_circular_mean(obs_s_p2, obs_c_p2, 'year')
    
    # Calculate Observed Difference (Shortest Path on Circle)
    # Result is in range [-pi, pi]
    obs_diff_rad = np.arctan2(np.sin(mean_angle_p2 - mean_angle_p1), 
                              np.cos(mean_angle_p2 - mean_angle_p1))
    
    # Convert to Absolute Days for the test statistic magnitude
    # We care about the MAGNITUDE of change for significance
    obs_stat = np.abs(obs_diff_rad)

    # --- 3. PERMUTATION LOOP ---
    # We count how many times random shuffled difference >= observed difference
    count_larger = xr.zeros_like(obs_stat)
    
    # Convert xarray to numpy for the loop to speed it up
    np_sin = sin_vals.values
    np_cos = cos_vals.values
    
    # Create array to hold shuffled indices
    indices = np.arange(n_total)
    
    for i in range(n_permutations):
        if i % 100 == 0: print(f"  Permutation {i}/{n_permutations}...", end='\r')
        
        # Shuffle indices
        np.random.shuffle(indices)
        
        # Split shuffled data
        idx_p1 = indices[:n_p1]
        idx_p2 = indices[n_p1:]
        
        # Slice numpy arrays
        # Assume year is the LAST axis for calculation.
        
        # Slice P1
        s_p1 = np.take(np_sin, idx_p1, axis=-1)
        c_p1 = np.take(np_cos, idx_p1, axis=-1)
        
        # Slice P2
        s_p2 = np.take(np_sin, idx_p2, axis=-1)
        c_p2 = np.take(np_cos, idx_p2, axis=-1)
        
        # Calculate Means (Mean of sin, Mean of cos)
        # axis=-1 matches the year dimension
        m_s_p1 = np.nanmean(s_p1, axis=-1)
        m_c_p1 = np.nanmean(c_p1, axis=-1)
        m_s_p2 = np.nanmean(s_p2, axis=-1)
        m_c_p2 = np.nanmean(c_p2, axis=-1)
        
        # Calculate Angles
        ang_p1 = np.arctan2(m_s_p1, m_c_p1)
        ang_p2 = np.arctan2(m_s_p2, m_c_p2)
        
        # Calculate Diff
        diff = np.arctan2(np.sin(ang_p2 - ang_p1), np.cos(ang_p2 - ang_p1))
        
        # Check Magnitude
        perm_stat = np.abs(diff)
        
        # Update Counter (using numpy where for speed)
        # We add 1 where the random difference is >= observed difference
        count_larger.values += (perm_stat >= obs_stat.values)

    print(f"  Permutation {n_permutations}/{n_permutations} Complete.")

    # --- 4. CALCULATE P-VALUE ---
    # (Count + 1) / (Permutations + 1) to avoid p=0
    p_values = (count_larger + 1) / (n_permutations + 1)
    
    # Return results
    # Convert observed difference back to days for plotting
    obs_diff_days = obs_diff_rad * (days_in_year / (2 * np.pi))
    
    return p_values, obs_diff_days

In [ ]:
# ==========================================
#  CALCULATE SIGNIFICANCE (P-VALUES)
# ==========================================

# IMPORTANT: CALL THIS NP.RANDOM.SEED(42) COMMAND EVERY TIME YOU RUN A SIGNIFICANCE TEST THAT USES NP.RANDOM.SHUFFLE
# TO ENSURE THE PERMUTATION SHUFFLES ARE EXACTLY REPRODUCIBLE.

np.random.seed(42)


# Run your circular permutation tests to get the p-values for S1 and S2
pval_s1_start, _ = circular_permutation_test(ds1['s1_start'], ds2['s1_start'])
np.random.seed(42)
pval_s2_start, _ = circular_permutation_test(ds1['s2_start'], ds2['s2_start'])


# For bimodal pixels, we consider the "average shift" significant if 
# at least one of the individual seasons showed a significant shift (see Methods in manuscript)
# Using np.fmin takes the lower p-value of the two.
combined_pval_start = xr.where(shift_s2_start.notnull(), np.fmin(pval_s1_start, pval_s2_start), pval_s1_start)


# Repeat for Ends
np.random.seed(42)
pval_s1_end, _ = circular_permutation_test(ds1['s1_end'], ds2['s1_end'])
np.random.seed(42)
pval_s2_end, _ = circular_permutation_test(ds1['s2_end'], ds2['s2_end'])

combined_pval_end = xr.where(shift_s2_end.notnull(), np.fmin(pval_s1_end, pval_s2_end), pval_s1_end)

In [ ]:
# AT THIS POINT WE HAVE TESTED SIGNIFICANCE OF THE START AND END DATE CHANGES
# NOW WE RUN THE FDR PROCEDURE

In [ ]:
# FDR Analysis

In [ ]:
def apply_fdr_control(p_map, alpha=0.1, method='indep'):
    """
    Applies the False Discovery Rate (FDR) control to a map of p-values
    following the Benjamini-Hochberg procedure (Wilks, 2016).
    
    Parameters:
    - p_map: xarray DataArray of p-values (lat x lon)
    - alpha: The global significance level (e.g., 0.05)
    - method: 'indep' (assuming independence/weak correlation) or 'dep' (strong correlation)
              Wilks (2016) suggests 'indep' is usually sufficient for climate data.
              
    Returns:
    - sig_mask: Boolean xarray (True where significant)
    """
    print(f"Applying FDR Control (alpha={alpha})...")
    
    # 1. Flatten the map to 1D array
    # We must mask NaNs (ocean/missing data) so they don't count towards N
    p_values = p_map.values.flatten()
    valid_mask = ~np.isnan(p_values)
    p_valid = p_values[valid_mask]
    
    N = len(p_valid) # Total number of tests
    
    # 2. Sort P-values (smallest to largest)
    sorted_p = np.sort(p_valid)
    
    # 3. Calculate Critical Values
    # Formula: (k / N) * alpha
    # k is 1-based index (1, 2, ..., N)
    k = np.arange(1, N + 1)
    
    if method == 'dep':
        # Benjamini-Yekutieli (for strong negative correlations)
        # Usually too conservative for spatial fields
        c_N = np.sum(1 / k)
        p_crit = (k / (N * c_N)) * alpha
    else:
        # Benjamini-Hochberg (Standard)
        p_crit = (k / N) * alpha
        
    # 4. Find Cutoff
    # Find the largest k where p_sorted[k] <= p_crit[k]
    # We check the condition (p <= crit)
    is_below = sorted_p <= p_crit
    
    if np.any(is_below):
        # The largest k is the last True value in the sorted array
        # We find the max index where this is true
        max_k_idx = np.where(is_below)[0].max()
        p_threshold = sorted_p[max_k_idx]
        
        print(f"  > FDR Threshold found: p <= {p_threshold:.5f}")
        print(f"  > (Standard p=0.05 threshold would be less strict)")
    else:
        print("  > No p-values passed FDR control.")
        p_threshold = 0.0
        
    # 5. Create Significance Mask
    sig_mask = p_map <= p_threshold
    
    return sig_mask

In [ ]:
# --- Apply FDR Control ---
# This gives you Boolean masks (True/False) instead of raw floats
sig_start_mask = apply_fdr_control(combined_pval_start, alpha=0.05)
sig_end_mask   = apply_fdr_control(combined_pval_end,   alpha=0.05)

In [ ]:
# NOW PLOT THE CHANGE IN START DATE AND END DATE

In [ ]:
def plot_single_change_fdr(differences, sig_mask, title, label, cmap='RdBu_r'):
    """
    Plots the change for a single variable, masking non-significant pixels
    and masking the ocean.
    """

    # 1. Apply FDR Mask
    diff_masked = differences.where(sig_mask)

    # 2. Plotting
    fig = plt.figure(figsize=(12, 7))
    ax = plt.axes(projection=ccrs.PlateCarree())
    
    # Plot Data (zorder=1)
    diff_masked.plot(
        ax=ax, transform=ccrs.PlateCarree(),
        cmap=cmap, center=0,
        #levels=np.arange(-90, 100, 10),
        levels=np.arange(-60, 70, 10),
        cbar_kwargs={'orientation': 'horizontal','label': 'Days', 'aspect': 30, 'pad': 0.08},
        extend='both',
        robust=True, zorder=1
    )

    # --- 3. MASK OCEAN (zorder=2) ---
    # We place this right after data plotting so it sits directly above the data layer
    ax.add_feature(cfeature.OCEAN, color='white', zorder=2)
    
    # --- 4. LAND BOUNDARIES & DECORATIONS (zorder=3) ---
    ax.coastlines(color='black', linewidth=0.8, zorder=3)
    ax.add_feature(cfeature.COASTLINE, linestyle=':', alpha=0.5, zorder=3)

    # --- 5. GRIDLINES (zorder=4) ---
    # Initializing at the absolute end forces them on top of the ocean polygon layer
    gl = ax.gridlines(
        draw_labels=True, 
        xlocs=np.arange(-180, 181, 60), 
        ylocs=np.arange(-90, 91, 30), 
        color='lightgray', 
        linewidth=0.8, 
        linestyle='-', 
        zorder=4
    )
    
    # Enforce labels location configuration
    gl.top_labels = False
    gl.right_labels = False
    
    # Explicitly force the underlying gridline collection artists to respect the zorder assignment
    gl.zorder = 4

    plt.title(title, fontsize=14)
    #plt.savefig("plots/ERA5_TMAX_SeasonStart_Change.pdf", format="pdf", bbox_inches="tight")
    plt.savefig("plots/ERA5_TMAX_SeasonEnd_Change.pdf", format="pdf", bbox_inches="tight")
    #plt.savefig("plots/ERA5_TMIN_SeasonStart_Change.pdf", format="pdf", bbox_inches="tight")
    #plt.savefig("plots/ERA5_TMIN_SeasonEnd_Change.pdf", format="pdf", bbox_inches="tight")
    
    # Force the map's boundary box to render on top of the gridlines
    ax.spines['geo'].set_zorder(5)
    plt.show()

In [ ]:
# Start Dates Map
plot_single_change_fdr(
    final_start_shift, 
    sig_start_mask, 
    title="Change (1996-2025) - (1966-1995) in TMAX Extreme Heat Season Start Date",
    label="Days Shifted (Positive = Later, Negative = Earlier)",
    cmap="RdBu" 
)

In [ ]:
# End Dates Map
plot_single_change_fdr(
    final_end_shift, 
    sig_end_mask, 
    title="Change (1996-2025) - (1966-1995) in TMAX Extreme Heat Season End Date",
    label="Days Shifted (Positive = Later, Negative = Earlier)",
    cmap="RdBu_r" # Often reversed for ends so red still means "worse/longer" 
)

In [ ]:
# NOW PLOT A CLASSIFICATION MAP THAT SHOWS WHETHER A GRID CELL HAD A START/END/BOTH CHANGE

In [ ]:
def plot_final_map_fdr(mask_start, mask_end, title="FDR-Significant Shifts in Heat Season"):
    """
    Plots the categorical map (Start Only, End Only, Both) using Boolean FDR masks.
    """
    print("Generating Final FDR Categorical Plot")
    
    fig = plt.figure(figsize=(12, 7))
    ax = plt.axes(projection=ccrs.PlateCarree())
    
    # -------------------------------------------------------------------------
    # 1. DEFINE CATEGORIES
    # -------------------------------------------------------------------------
    # 0: None
    # 1: Start Only
    # 2: End Only
    # 3: Both
    
    # Initialize with zeros
    cat = xr.zeros_like(mask_start, dtype=int)
    
    # Logic: Where is the mask True?
    cat = xr.where((mask_start) & (~mask_end), 1, cat) # Start Only
    cat = xr.where((~mask_start) & (mask_end), 2, cat) # End Only
    cat = xr.where((mask_start) & (mask_end), 3, cat)  # Both
    
    
    # -------------------------------------------------------------------------
    # 2. PLOT CATEGORIES
    # -------------------------------------------------------------------------
    # Colors: 0=White, 1=Orange, 2=Purple, 3=Red
    cmap = mcolors.ListedColormap(['white', 'orange', 'purple', 'red'])
    norm = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)
    
    # Plot Data (zorder=1)
    cat.plot(ax=ax, transform=ccrs.PlateCarree(), cmap=cmap, norm=norm, 
             add_colorbar=False, zorder=1)
    
    # -------------------------------------------------------------------------
    # 3. MASK OCEAN & DECORATION
    # -------------------------------------------------------------------------
    # Mask Ocean (White layer on top of data)
    ax.add_feature(cfeature.OCEAN, color='white', zorder=2)
    
    # Coastlines & Borders
    ax.coastlines(linewidth=0.8, color='black', zorder=3)
    ax.add_feature(cfeature.COASTLINE, linestyle=':', alpha=0.5, zorder=3)
    
    # Lat/Lon Lines (20N / 20S)
    #ax.hlines([-20, 20], -180, 180, 'k', '--', lw=0.5, transform=ccrs.PlateCarree(), zorder=3)

    # Gridlines
    # Set zorder to 4 so they appear over the white ocean mask
    gl = ax.gridlines(
        draw_labels=True, 
        xlocs=np.arange(-180, 181, 60), 
        ylocs=np.arange(-90, 91, 30), 
        color='lightgray', 
        linewidth=0.8, 
        linestyle='-', 
        zorder=4
    )
    
    # This forces the labels to only appear on the bottom and left
    gl.top_labels = False
    gl.right_labels = False
    
    # -------------------------------------------------------------------------
    # 4. LEGEND
    # -------------------------------------------------------------------------
    handles = [
        mpatches.Patch(color='white', label='Neither', ec='lightgray'),
        mpatches.Patch(color='orange', label='Start Only'),
        mpatches.Patch(color='purple', label='End Only'),
        mpatches.Patch(color='red', label='Both')
    ]
    ax.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.1), 
              ncol=4, frameon=False, fontsize=11)

    plt.title(title, fontsize=14, pad=15)
    plt.savefig("plots/ERA5_TMAX_SeasonShift_Classification.pdf", format="pdf", bbox_inches="tight")
    #plt.savefig("plots/ERA5_TMIN_SeasonShift_Classification.pdf", format="pdf", bbox_inches="tight")

    plt.show()

In [ ]:
plot_final_map_fdr(sig_start_mask, sig_end_mask, title="TMAX Extreme Heat Season Changes")
#plot_final_map_fdr(sig_start_mask, sig_end_mask, title="TMIN Extreme Heat Season Changes")